# REV: Full Training, Checkpoint Saving & Live API
### Real Datasets (AG News, BoolQ, SST-5) + Synthetic Policy Rules + LoRA + FastAPI

This notebook trains a complete decision model on real datasets, saves the trained weights (`head.pt` + LoRA adapter), and launches a live `POST /v1/systemone` API server.

In [ ]:
# 1. Install & upgrade dependencies
!pip install -q -U torchao peft transformers datasets accelerate pydantic fastapi uvicorn pyngrok huggingface_hub

In [ ]:
# 2. Core Architecture & Attention Masking
import os, math, re, time, random, json, hashlib
from typing import Any, Literal, Union, Dict, List
import torch
import torch.nn as nn
import torch.nn.functional as F
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

SPECIAL = ["<|fim_prefix|>", "<|fim_middle|>", "<|box_start|>", "<|box_end|>", "<|fim_suffix|>"]
MAX_STATE, MAX_BRANCH = 384, 1024
_SPECIAL_RE = re.compile(r"<\|([A-Za-z0-9_]+)\|>")

def user_tokens(tok, text: str):
    return tok(_SPECIAL_RE.sub(r"<¦\1¦>", str(text)), add_special_tokens=False).input_ids

OPT_NONE, OPT_DECIDE = -1, -2

def encode(tok, rec, max_state=MAX_STATE, max_branch=MAX_BRANCH):
    state_tokens = user_tokens(tok, rec["state"])
    S = [tok.convert_tokens_to_ids(SPECIAL[0])] + state_tokens[: max_state - 1]
    ids, seg, pos, opt = list(S), [0] * len(S), list(range(len(S))), [OPT_NONE] * len(S)
    q_id, o_id, c_id, d_id = [tok.convert_tokens_to_ids(t) for t in SPECIAL[1:]]
    decide_idx, opt_idx = [], []
    p0 = len(S)
    
    for k, q in enumerate(rec["questions"], start=1):
        instr = [q_id] + user_tokens(tok, q["instr"])
        spans = [[o_id] + user_tokens(tok, o) + [c_id] for o in q["options"]]
        br = instr + [t for sp in spans for t in sp] + [d_id]
        base = len(ids)
        br_pos = list(range(p0, p0 + len(br)))
        br_opt = [OPT_NONE] * len(instr) + [j for j, sp in enumerate(spans) for _ in sp] + [OPT_DECIDE]
        ends, cursor = [], len(instr)
        for sp in spans:
            cursor += len(sp)
            ends.append(cursor - 1)
        ids += br; seg += [k] * len(br); pos += br_pos; opt += br_opt
        decide_idx.append(base + len(br) - 1)
        opt_idx.append([base + e for e in ends])
        
    return {"ids": ids, "seg": seg, "pos": pos, "opt": opt, "decide_idx": decide_idx, "opt_idx": opt_idx,
            "labels": [q.get("label", 0) for q in rec["questions"]]}

# --- Prefix KV-Caching Functions ---
def encode_state(tok, state_text: str, max_state: int = MAX_STATE):
    state_tokens = user_tokens(tok, state_text)
    prefix_ids = [tok.convert_tokens_to_ids(SPECIAL[0])] + state_tokens[: max_state - 1]
    return {"ids": prefix_ids, "pos": list(range(len(prefix_ids))), "length": len(prefix_ids)}

def encode_question_branches(tok, questions: list[dict], state_len: int):
    q_id, o_id, c_id, d_id = [tok.convert_tokens_to_ids(t) for t in SPECIAL[1:]]
    branches = []
    for q in questions:
        instr = [q_id] + user_tokens(tok, q["instr"])
        spans = [[o_id] + user_tokens(tok, o) + [c_id] for o in q["options"]]
        br = instr + [t for sp in spans for t in sp] + [d_id]
        br_pos = list(range(state_len, state_len + len(br)))
        ends, cursor = [], len(instr)
        for sp in spans:
            cursor += len(sp)
            ends.append(cursor - 1)
        branches.append({"ids": br, "pos": br_pos, "decide_idx": len(br) - 1, "opt_idx": ends, "num_opts": len(spans)})
    max_len = max(len(b["ids"]) for b in branches) if branches else 0
    return {"branches": branches, "max_len": max_len}

def clone_or_expand_past_key_values(past_key_values, batch_size: int = 1):
    if past_key_values is None: return None
    from transformers.cache_utils import DynamicCache
    if hasattr(past_key_values, "layers") and len(past_key_values.layers) > 0:
        new_data = []
        for l in past_key_values.layers:
            k, v = l.keys, l.values
            if batch_size == 1:
                k_out, v_out = k.clone(), v.clone()
            else:
                k_out, v_out = k.expand(batch_size, -1, -1, -1).contiguous(), v.expand(batch_size, -1, -1, -1).contiguous()
            sw = getattr(l, "_sliding_window_tensor", None)
            new_data.append((k_out, v_out, sw) if sw is not None else (k_out, v_out))
        return DynamicCache(ddp_cache_data=new_data)
    elif hasattr(past_key_values, "key_cache") and hasattr(past_key_values, "value_cache"):
        new_c = DynamicCache()
        if batch_size == 1:
            new_c.key_cache = [k.clone() for k in past_key_values.key_cache]
            new_c.value_cache = [v.clone() for v in past_key_values.value_cache]
        else:
            new_c.key_cache = [k.expand(batch_size, -1, -1, -1).contiguous() for k in past_key_values.key_cache]
            new_c.value_cache = [v.expand(batch_size, -1, -1, -1).contiguous() for v in past_key_values.value_cache]
        if hasattr(past_key_values, "_seen_tokens"): new_c._seen_tokens = past_key_values._seen_tokens
        return new_c
    elif isinstance(past_key_values, (tuple, list)):
        if batch_size == 1: return tuple((k.clone(), v.clone()) for k, v in past_key_values)
        return tuple((k.expand(batch_size, -1, -1, -1).contiguous(), v.expand(batch_size, -1, -1, -1).contiguous()) for k, v in past_key_values)
    return past_key_values

def branch_mask_batch(segs, device, dtype=torch.float32):
    L = max(len(s) for s in segs)
    s = torch.full((len(segs), L), -1, device=device)
    for b, seg in enumerate(segs):
        s[b, : len(seg)] = torch.tensor(seg, device=device)
    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=device))
    same = (s[:, None, :] == s[:, :, None]) | (s[:, None, :] == 0)
    valid_key = (s != -1)[:, None, :]
    allow = (causal[None] & same & valid_key) | torch.eye(L, dtype=torch.bool, device=device)[None]
    mask = torch.zeros(len(segs), L, L, dtype=dtype, device=device)
    return mask.masked_fill(~allow, torch.finfo(dtype).min)[:, None]

class PointerHead(nn.Module):
    def __init__(self, d: int, dp: int = 256):
        super().__init__()
        self.q = nn.Linear(d, dp)
        self.k = nn.Linear(d, dp)
        self.scale = 1.0 / math.sqrt(dp)

    def forward(self, h_decide, h_opts):
        return (self.k(h_opts) @ self.q(h_decide)) * self.scale

class DecisionModel(nn.Module):
    def __init__(self, base_name="Qwen/Qwen2.5-0.5B", lora_r=16, head_dim=256, device="cuda", gradient_checkpointing=False):
        super().__init__()
        self.device = device
        self.base_name = base_name
        self.lora_r = lora_r
        self.head_dim = head_dim
        dtype = torch.bfloat16 if ("cuda" in str(device) and torch.cuda.is_bf16_supported()) else (torch.float16 if "cuda" in str(device) else torch.float32)
        self.lm = AutoModelForCausalLM.from_pretrained(
            base_name,
            attn_implementation="sdpa" if "cuda" in str(device) else "eager",
            torch_dtype=dtype,
            device_map=device if "cuda" in str(device) else None,
            low_cpu_mem_usage=True,
        ).model
        if gradient_checkpointing:
            self.lm.gradient_checkpointing_enable()
        if lora_r > 0:
            cfg = LoraConfig(task_type="FEATURE_EXTRACTION", r=lora_r, lora_alpha=2 * lora_r,
                             lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
            self.lm = get_peft_model(self.lm, cfg)
        self.head = PointerHead(self.lm.config.hidden_size, dp=head_dim).to(device)
        self.to(device)

    def hidden_batch(self, encs):
        L = max(len(e["ids"]) for e in encs)
        ids = torch.full((len(encs), L), 0, device=self.device)
        pos = torch.zeros((len(encs), L), dtype=torch.long, device=self.device)
        for b, e in enumerate(encs):
            ids[b, : len(e["ids"])] = torch.tensor(e["ids"], device=self.device)
            pos[b, : len(e["pos"])] = torch.tensor(e["pos"], device=self.device)
        lm_dtype = next(self.lm.parameters()).dtype
        mask = branch_mask_batch([e["seg"] for e in encs], self.device, dtype=lm_dtype)
        return self.lm(input_ids=ids, position_ids=pos, attention_mask=mask).last_hidden_state.float()

    def forward_batch(self, encs):
        hs = self.hidden_batch(encs)
        return [[self.head(hs[b][d], hs[b][torch.tensor(oi, device=self.device)])
                 for d, oi in zip(e["decide_idx"], e["opt_idx"])] for b, e in enumerate(encs)]

    @torch.no_grad()
    def probs(self, enc):
        logits = self.forward_batch([enc])[0]
        return [F.softmax(z, dim=-1).cpu() for z in logits]

    @torch.no_grad()
    def compute_state_cache(self, tok, state_text: str, max_state: int = MAX_STATE):
        enc_state = encode_state(tok, state_text, max_state=max_state)
        ids = torch.tensor([enc_state["ids"]], device=self.device, dtype=torch.long)
        pos = torch.tensor([enc_state["pos"]], device=self.device, dtype=torch.long)
        out = self.lm(input_ids=ids, position_ids=pos, use_cache=True)
        h = hashlib.sha256(state_text.strip().encode("utf-8")).hexdigest()
        return out.past_key_values, enc_state["length"], h

    @torch.no_grad()
    def forward_with_cache(self, branch_data: dict, past_key_values, state_len: int):
        branches = branch_data["branches"]
        if not branches: return []
        B, L_max = len(branches), branch_data["max_len"]
        ids = torch.zeros((B, L_max), dtype=torch.long, device=self.device)
        pos = torch.zeros((B, L_max), dtype=torch.long, device=self.device)
        lm_dtype = next(self.lm.parameters()).dtype
        K_total = state_len + L_max
        mask_2d = torch.zeros((B, K_total), dtype=torch.long, device=self.device)
        mask_2d[:, :state_len] = 1
        for b, br in enumerate(branches):
            l = len(br["ids"])
            ids[b, :l] = torch.tensor(br["ids"], device=self.device)
            pos[b, :l] = torch.tensor(br["pos"], device=self.device)
            mask_2d[b, state_len : state_len + l] = 1
        expanded_pkv = clone_or_expand_past_key_values(past_key_values, batch_size=B)
        out = self.lm(input_ids=ids, position_ids=pos, attention_mask=mask_2d, past_key_values=expanded_pkv)
        hs = out.last_hidden_state.float()
        return [self.head(hs[b, br["decide_idx"]], hs[b, torch.tensor(br["opt_idx"], device=self.device)]) for b, br in enumerate(branches)]

    @torch.no_grad()
    def probs_cached(self, tok, rec: dict, past_key_values, state_len: int):
        branch_data = encode_question_branches(tok, rec["questions"], state_len)
        logits = self.forward_with_cache(branch_data, past_key_values, state_len)
        return [F.softmax(z, dim=-1).cpu() for z in logits]


In [ ]:
# 3. API Schemas & Data Types
JSONContent = Union[str, dict, list, int, float, bool, None]

class Noul(BaseModel):
    type: Literal["noul"] = "noul"
    instructions: JSONContent
    criteria: dict[str, JSONContent] | None = None

class Choice(BaseModel):
    type: Literal["choice"] = "choice"
    instructions: JSONContent
    criteria: dict[str, JSONContent]

class Score(BaseModel):
    type: Literal["score"] = "score"
    instructions: JSONContent
    criteria: list[JSONContent] = Field(min_length=2, max_length=255)

class SystemOneRequest(BaseModel):
    state: JSONContent
    model: str = "rev-latest"
    questions: dict[str, Union[Noul, Choice, Score]]

def render(v: JSONContent, indent: int = 0) -> str:
    pad = "  " * indent
    if v is None: return ""
    if isinstance(v, (str, int, float, bool)): return str(v)
    if isinstance(v, list): return "\n".join(f"{pad}- {render(x, indent + 1).lstrip()}" for x in v)
    return "\n".join(f"{pad}{k}:\n{render(x, indent + 1)}" if isinstance(x, (dict, list)) else f"{pad}{k}: {render(x)}" for k, x in v.items())

def to_record(req: SystemOneRequest):
    qs, meta = [], []
    for qid, q in req.questions.items():
        instr = render(q.instructions)
        if q.type == "noul":
            c = q.criteria or {}
            opts = [f"no: {render(c.get('false'))}" if c.get('false') else "no",
                    f"yes: {render(c.get('true'))}" if c.get('true') else "yes"]
            meta.append({"id": qid, "type": "noul"})
        elif q.type == "choice":
            opts = [f"{k}: {render(v)}" if v else k for k, v in q.criteria.items()]
            meta.append({"id": qid, "type": "choice", "keys": list(q.criteria.keys())})
        else:
            opts = [render(x) for x in q.criteria]
            meta.append({"id": qid, "type": "score", "legend": {str(i): render(x) for i, x in enumerate(q.criteria)}})
        qs.append({"instr": instr, "options": opts, "label": 0})
    return {"state": render(req.state), "questions": qs}, meta

def to_answers(probs: list[list[float]], meta: list[dict]) -> dict[str, Any]:
    out = {}
    for p, m in zip(probs, meta):
        if m["type"] == "noul":
            out[m["id"]] = {"type": "noul", "noul": round(p[1], 3)}
        elif m["type"] == "choice":
            best_idx = max(range(len(p)), key=lambda i: p[i])
            conf = 1.0 if len(p) == 1 else (max(p) - 1.0 / len(p)) / (1.0 - 1.0 / len(p))
            out[m["id"]] = {"type": "choice", "choice": m["keys"][best_idx], "confidence": round(conf, 3),
                            "probabilities": {k: round(v, 4) for k, v in zip(m["keys"], p)}}
        else:
            score = sum(i * pi for i, pi in enumerate(p))
            mode = max(range(len(p)), key=lambda i: p[i])
            conf = 1.0 - sum(pi * abs(i - mode) for i, pi in enumerate(p)) / (len(p) - 1)
            out[m["id"]] = {"type": "score", "score": round(score, 2), "confidence": round(conf, 3),
                            "legend": m["legend"], "probabilities": {str(i): round(v, 4) for i, v in enumerate(p)}}
    return out

In [ ]:
# 4. Real Datasets Loader (AG News, BoolQ, SST-5, + Synthetic Contrastive Pairs)
from datasets import load_dataset

def build_dataset(n_samples_per_task=200):
    dataset = []
    print(f"Downloading real datasets ({n_samples_per_task} samples per source)...")
    
    # Task 1: AG News (Choice: 4 topics)
    try:
        ag = load_dataset("fancyzhx/ag_news", split="train").shuffle(seed=42).select(range(n_samples_per_task))
        ag_topics = ["world", "sports", "business", "scitech"]
        for row in ag:
            dataset.append({
                "state": row["text"],
                "questions": [{
                    "instr": "What is the primary topic of this news report?",
                    "options": ["World politics and international news", "Sports games and athletes", "Business and finance", "Science and technology"],
                    "label": row["label"],
                    "qtype": "choice"
                }]
            })
        print(f"✓ Loaded {len(ag)} AG News samples")
    except Exception as e: print("Skipping AG News:", e)

    # Task 2: BoolQ (Noul: Yes/No reading comprehension)
    try:
        boolq = load_dataset("google/boolq", split="train").shuffle(seed=42).select(range(n_samples_per_task))
        for row in boolq:
            dataset.append({
                "state": row["passage"],
                "questions": [{
                    "instr": f"Question: {row['question']}?",
                    "options": ["no", "yes"],
                    "label": 1 if row["answer"] else 0,
                    "qtype": "noul"
                }]
            })
        print(f"✓ Loaded {len(boolq)} BoolQ samples")
    except Exception as e: print("Skipping BoolQ:", e)

    # Task 3: SST-5 (Score: 5 sentiment rating levels)
    try:
        sst5 = load_dataset("SetFit/sst5", split="train").shuffle(seed=42).select(range(n_samples_per_task))
        sst_options = ["very negative", "negative", "neutral", "positive", "very positive"]
        for row in sst5:
            dataset.append({
                "state": row["text"],
                "questions": [{
                    "instr": "What is the sentiment of this text?",
                    "options": sst_options,
                    "label": row["label"],
                    "qtype": "score"
                }]
            })
        print(f"✓ Loaded {len(sst5)} SST-5 samples")
    except Exception as e: print("Skipping SST-5:", e)

    # Task 4: Synthetic Policy Minimal Pairs (Prevents 'None of the above' shortcut)
    policies = [
        ("Refund policy: Full refund allowed if returned within 30 days in original packaging.",
         "Item was returned after 45 days in original packaging. Is refund approved?", 0),
        ("Refund policy: Full refund allowed if returned within 30 days in original packaging.",
         "Item was returned after 12 days in original packaging. Is refund approved?", 1),
        ("Security rule: Access level 3 requires MFA and active VPN.",
         "User has MFA enabled but no VPN active. Allow access?", 0),
        ("Security rule: Access level 3 requires MFA and active VPN.",
         "User has MFA enabled and active corporate VPN. Allow access?", 1),
    ] * 25
    for state, q_text, label in policies:
        dataset.append({
            "state": state,
            "questions": [{"instr": q_text, "options": ["no", "yes"], "label": label, "qtype": "noul"}]
        })
    print(f"✓ Added {len(policies)} Synthetic Policy Pairs")
    
    random.shuffle(dataset)
    print(f"Total training records prepared: {len(dataset)}")
    return dataset

train_data = build_dataset(n_samples_per_task=150)

In [ ]:
# 5. Train with LoRA (Accumulation, lr=5e-5, Ordinal RPS Loss)
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ====================================================================
# MODEL CONFIGURATION: SCALE BACKBONE & PARAMETERS HERE
# ====================================================================
# Available backbones:
# - "Qwen/Qwen2.5-0.5B"     -> 0.5B params (~1.0 GB VRAM, ultra fast)
# - "Qwen/Qwen3-0.6B-Base"   -> 0.6B params (~1.5 GB VRAM, Qwen3 baseline)
# - "Qwen/Qwen2.5-1.5B"     -> 1.5B params (~3.5 GB VRAM, 3x larger)
# - "Qwen/Qwen2.5-3B"       -> 3.0B params (~6.0 GB VRAM, 6x larger)
# - "Qwen/Qwen3-4B-Base"     -> 4.0B params (~8.5 GB VRAM - Jared Palmer's Best!)
# - "Qwen/Qwen3-8B-Base"     -> 8.0B params (~14.5 GB VRAM in fp16)
# - "Qwen/Qwen2.5-7B"       -> 7.0B params (~14.0 GB VRAM in fp16)

BASE_NAME = "Qwen/Qwen3-4B-Base"    # <-- Change this to scale parameters!
LORA_R = 16                        # LoRA Rank (16, 32, or 64)
HEAD_DIM = 256                     # PointerHead projection dimension (256 or 512)
GRAD_CHECKPOINT = True if any(x in BASE_NAME for x in ["4B", "7B", "8B"]) else False
ACCUM_STEPS = 8 if any(x in BASE_NAME for x in ["4B", "7B", "8B"]) else 4
LR = 5e-5                          # Optimal learning rate (preserves base capabilities)
EPOCHS = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing {BASE_NAME} on: {device} (Gradient Checkpointing: {GRAD_CHECKPOINT})")

tok = AutoTokenizer.from_pretrained(BASE_NAME)
model = DecisionModel(
    base_name=BASE_NAME,
    lora_r=LORA_R,
    head_dim=HEAD_DIM,
    device=device,
    gradient_checkpointing=GRAD_CHECKPOINT
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
model.train()

for epoch in range(EPOCHS):
    total_loss, count = 0.0, 0
    optimizer.zero_grad()
    for i, sample in enumerate(train_data):
        enc = encode(tok, sample)
        logits_list = model.forward_batch([enc])[0]
        
        loss = 0.0
        for z, q in zip(logits_list, sample["questions"]):
            y = torch.tensor([q["label"]], device=device)
            ce = F.cross_entropy(z[None], y)
            if q["qtype"] == "score":
                p = F.softmax(z, dim=-1)
                cdf = (torch.arange(len(p) - 1, device=device) >= q["label"]).to(p.dtype)
                rps = (p.cumsum(-1)[:-1] - cdf).square().mean()
                ce = ce + 0.5 * rps
            loss = loss + ce
            
        (loss / ACCUM_STEPS).backward()
        total_loss += loss.item()
        count += 1
        
        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(train_data):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            
        if (i + 1) % 100 == 0:
            print(f"Epoch {epoch+1} | Step {i+1}/{len(train_data)} | Loss: {total_loss/count:.4f}")
            
    print(f"--> Epoch {epoch+1} Completed! Average Loss: {total_loss / count:.4f}")

print(f"Training Complete on {BASE_NAME}!")


In [ ]:
# 6. Save Model Checkpoint (LoRA Adapter + Pointer Head + Tokenizer)
SAVE_DIR = "./rev_checkpoint"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save LoRA adapter weights
model.lm.save_pretrained(SAVE_DIR)

# 2. Save PointerHead and architecture configuration
torch.save({
    "head": model.head.state_dict(),
    "base": BASE_NAME,
    "lora": LORA_R,
    "head_dim": HEAD_DIM,
}, f"{SAVE_DIR}/head.pt")

# 3. Save tokenizer configurations
tok.save_pretrained(SAVE_DIR)

print(f"Checkpoint saved successfully in {SAVE_DIR} (Base: {BASE_NAME})")


In [ ]:
# 7. Launch Live FastAPI Server (POST /v1/systemone)
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import threading, uvicorn

app = FastAPI(title="jev")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.post("/v1/systemone")
def systemone(req: SystemOneRequest):
    rec, meta = to_record(req)
    enc = encode(tok, rec)
    t0 = time.time()
    ps = model.probs(enc)
    latency_ms = round((time.time() - t0) * 1000, 2)
    answers = to_answers([p.tolist() for p in ps], meta)
    return {
        "model": req.model,
        "answers": answers,
        "usage": {"tokens": len(enc["ids"])},
        "latency_ms": latency_ms
    }

@app.get("/v1/models")
def models():
    return {"models": [{"id": "rev-latest", "base": "Qwen/Qwen2.5-0.5B"}]}

# Run FastAPI in a background thread inside Colab
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print("✓ FastAPI server is running on http://127.0.0.1:8000!")

In [ ]:
# 8. Test the Live API with a Complex Multi-Question Request
import requests

payload = {
    "state": "The company announced quarterly revenues of $14.2B, exceeding analyst expectations by 8%. However, operating margins decreased due to increased AI infrastructure capital expenditures.",
    "model": "rev-latest",
    "questions": {
        "topic": {
            "type": "choice",
            "instructions": "What is the primary topic of this excerpt?",
            "criteria": {
                "sports": "Athletics, games, and teams",
                "finance": "Corporate earnings, revenue, and markets",
                "world": "International politics",
                "scitech": "General consumer gadgets"
            }
        },
        "positive_earnings": {
            "type": "noul",
            "instructions": "Did the revenue beat analyst expectations?"
        },
        "sentiment": {
            "type": "score",
            "instructions": "What is the overall sentiment of this financial update?",
            "criteria": ["very negative", "negative", "neutral", "positive", "very positive"]
        }
    }
}

response = requests.post("http://127.0.0.1:8000/v1/systemone", json=payload)
print("Status Code:", response.status_code)
print(json.dumps(response.json(), indent=2))

In [ ]:
# 9. Benchmark: State Prefix KV-Caching (<5ms repeated latency on long documents)
# In rev, we cache key/value activations of the document in GPU memory.
# Multiple subsequent questions skip document prefill entirely!
model.__class__ = DecisionModel  # Bind updated class methods to existing instance

long_doc = (
    "EMPLOYMENT AGREEMENT\n"
    "This Employment Agreement is entered into as of October 1, 2026, between Rev Labs Inc. ('Company') "
    "and Jane Doe ('Executive').\n"
    "1. Base Salary: Annual base salary of $350,000, payable semi-monthly.\n"
    "2. Equity Incentive: Executive is granted 100,000 RSUs vesting over 4 years with a 1-year cliff.\n"
    "3. Non-Compete: Executive agrees not to engage in competing business for 12 months post-termination.\n"
    "4. Severance: In event of termination without Cause, Company pays 6 months base salary.\n"
    "5. Governing Law: State of California, County of San Francisco."
)

print('=== STEP 1: Cold Document Prefill ===')
t0 = time.time()
pkv, s_len, s_hash = model.compute_state_cache(tok, long_doc)
t_cold = (time.time() - t0) * 1000
print(f'Document prefill computed: {t_cold:.2f} ms ({s_len} tokens, SHA: {s_hash[:10]}...)')

questions = [
    {
        'instr': 'What is the executive annual base salary?',
        'options': ['$250,000', '$350,000', '$500,000'],
    },
    {
        'instr': 'What is the duration of the non-compete covenant?',
        'options': ['6 months', '12 months', '24 months'],
    },
    {
        'instr': 'How many months of severance are guaranteed upon termination without cause?',
        'options': ['3 months', '6 months', '12 months'],
    },
]

print('\n=== STEP 2: Subsequent Questions against Cached State (Sub-5ms Target) ===')
for i, q in enumerate(questions, start=1):
    sub_rec = {'state': long_doc, 'questions': [q]}
    t0 = time.time()
    ps = model.probs_cached(tok, sub_rec, pkv, s_len)
    t_cached = (time.time() - t0) * 1000
    best_idx = ps[0].argmax().item()
    print(f'Q{i}: "{q["instr"]}"')
    print(f'   Decision: {q["options"][best_idx]} ({ps[0].max().item():.1%})')
    print(f'   Latency:  {t_cached:.2f} ms (Speedup: {t_cold / max(t_cached, 0.001):.1f}x faster)\n')

print('State Prefix KV-Caching verified: Zero document recomputation on repeat queries!')


In [ ]:
# 10. Publish Trained Model to Hugging Face Hub
from huggingface_hub import login, HfApi

# 1. Log in to Hugging Face (Paste your Write Token from https://huggingface.co/settings/tokens)
login()

# 2. Specify your Hugging Face username and repository name
HF_USERNAME = "your-username"  # <-- Replace with your Hugging Face username!
tag = BASE_NAME.split("/")[-1].lower().replace("qwen-", "").replace("qwen2.5-", "").replace("-base", "")
REPO_NAME = f"{HF_USERNAME}/rev-{tag}"

# 3. Create Model Card (README.md)
model_card = f"""---
base_model: {BASE_NAME}
base_model_relation: adapter
library_name: peft
tags:
- decision-model
- jev
- typesafe
- lora
- prefill-only
---

# {REPO_NAME}

A Rev-style prefill-only decision model based on `Qwen/Qwen2.5-0.5B`.

### Features
- **Single forward pass**: Answers multiple questions about a document in parallel with zero autoregressive decoding.
- **Block-causal attention**: Strict branch isolation so questions do not leak into each other.
- **Pointer readout head**: Bilinear projection over option boundaries.
- **API**: TypeSafe System One contract (`POST /v1/systemone`).

### Files included
- `adapter_model.safetensors`: Trained LoRA adapter
- `head.pt`: PointerHead weights and base model metadata
- Tokenizer configurations
"""

with open(f"{SAVE_DIR}/README.md", "w") as f:
    f.write(model_card)

# 4. Upload folder to Hugging Face Hub
api = HfApi()
print(f"Creating repo: {REPO_NAME}...")
api.create_repo(repo_id=REPO_NAME, repo_type="model", exist_ok=True)

print(f"Uploading checkpoint files from {SAVE_DIR} to https://huggingface.co/{REPO_NAME}...")
info = api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=REPO_NAME,
    repo_type="model",
    commit_message="Upload trained rev-0.5b decision model adapter and pointer head"
)

print(f"\n✓ Model published successfully to: https://huggingface.co/{REPO_NAME}")